# Booty Hunter - Dev Build Compiler
Downloads the game folder, compiles with the REAL Ren'Py 8.5.2 SDK (native x86_64, no emulation), repackages `0script.rpa`, uploads the result back to GitHub as a release asset.

In [ ]:
# 1) Download Ren'Py SDK (matches game's Ren'Py 8.5.2.26010301)
!wget -q "https://www.renpy.org/dl/8.5.2/renpy-8.5.2-sdk.tar.bz2" -O /content/renpy_sdk.tar.bz2
!tar xjf /content/renpy_sdk.tar.bz2 -C /content
!ls /content | grep renpy


In [ ]:
# 2) Download the game folder (edited .rpy files already included, all .rpyc deleted for a clean 00_membershipInstallation/99_developer_override, rest intact)
!wget -q "https://github.com/m7830380-cyber/boothunter-devbuild/releases/download/game-v1/colab_game.zip" -O /content/colab_game.zip
!mkdir -p /content/gamebuild
!cd /content/gamebuild && unzip -q /content/colab_game.zip
!mv /content/gamebuild/colab_game /content/gamebuild/game
!ls /content/gamebuild/game


In [ ]:
# 3) Compile with the real SDK (this generates/updates .rpyc for ALL .rpy files, including our 2 edited ones)
import glob, os
sdk_dir = glob.glob('/content/renpy-*-sdk')[0]
print("SDK:", sdk_dir)
!{sdk_dir}/renpy.sh /content/gamebuild compile


In [ ]:
# 4) Sanity check: confirm our files compiled and no errors were raised
import os
mi = "/content/gamebuild/game/scripts/Classes/00_membershipInstallation.rpyc"
do = "/content/gamebuild/game/scripts/Classes/99_developer_override.rpyc"
print("membershipInstallation.rpyc exists:", os.path.isfile(mi), os.path.getsize(mi) if os.path.isfile(mi) else 0)
print("developer_override.rpyc exists:", os.path.isfile(do), os.path.getsize(do) if os.path.isfile(do) else 0)
!cat /content/gamebuild/game/log.txt 2>/dev/null | tail -40


In [ ]:
# 5) Repackage into 0script.rpa (rpatool, official Shizmob tool)
!git clone -q https://github.com/Shizmob/rpatool.git /content/rpatool
!cd /content/gamebuild/game && python3 /content/rpatool/rpatool -c /content/0script.rpa .
!ls -lh /content/0script.rpa


In [ ]:
# 6) Upload result back to GitHub release so it can be pulled into the phone session
import os
token = os.environ.get("GITHUB_TOKEN") or input("Paste your GITHUB_TOKEN (same one used on the phone): ")
os.environ["GH_TOKEN"] = token
!pip -q install PyGithub >/dev/null 2>&1
!wget -q https://github.com/cli/cli/releases/download/v2.59.0/gh_2.59.0_linux_amd64.tar.gz -O /content/gh.tar.gz
!tar xzf /content/gh.tar.gz -C /content
GH_BIN = "/content/gh_2.59.0_linux_amd64/bin/gh"
!{GH_BIN} auth status 2>&1 | head -5
!{GH_BIN} release upload game-v1 /content/0script.rpa --repo m7830380-cyber/boothunter-devbuild --clobber
print("Uploaded. Download with:")
print("https://github.com/m7830380-cyber/boothunter-devbuild/releases/download/game-v1/0script.rpa")
